In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit.primitives import StatevectorEstimator

In [ ]:
# ---------------------------------------------------------------------------
# Hamiltonian H = X ⊗ Y
# Qiskit's SparsePauliOp string is little-endian: "XY" → X on q1, Y on q0
# which corresponds to the tensor product X(q1) ⊗ Y(q0) = X ⊗ Y in the
# standard |q1 q0⟩ ket ordering. Ground state energy is -1.0.
# ---------------------------------------------------------------------------

In [ ]:
H = SparsePauliOp("XY")

In [ ]:
# Print exact spectrum for reference
eigvals = np.linalg.eigvalsh(Operator(H).data)
print(f"Exact eigenvalues: {np.round(eigvals, 6)}")
print(f"Exact ground state energy: {eigvals[0]:.6f}\n")

In [ ]:
# ---------------------------------------------------------------------------
# Ansatz: product state — Ry then Rz on each qubit independently.
# Any single-qubit pure state is reachable with Ry(α)·Rz(β)|0⟩,
# which is sufficient since the ground state is unentangled.
# ---------------------------------------------------------------------------
theta = ParameterVector("θ", 4)   # θ0,θ1 for q0;  θ2,θ3 for q1
ansatz = QuantumCircuit(2)
ansatz.ry(theta[0], 0)
ansatz.rz(theta[1], 0)
ansatz.ry(theta[2], 1)
ansatz.rz(theta[3], 1);

In [ ]:
print("Ansatz:")
print(ansatz.draw(), "\n")

In [ ]:
estimator = StatevectorEstimator()

In [ ]:
# ---------------------------------------------------------------------------
# Cost function: ⟨ψ(θ)|H|ψ(θ)⟩
# ---------------------------------------------------------------------------

In [ ]:
def cost_func(params):
    job = estimator.run([(ansatz, H, [params])])
    return float(job.result()[0].data.evs[0])

In [ ]:
# ---------------------------------------------------------------------------
# Optimize with multiple random restarts to escape local minima
# ---------------------------------------------------------------------------

In [ ]:
np.random.seed(42)
best_result = None

for trial in range(8):
    x0 = np.random.uniform(0, 2 * np.pi, size=4)
    res = minimize(cost_func, x0, method="cobyla", options={"maxiter": 2000, "rhobeg": 0.5})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

In [ ]:
print(f"Optimized ground state energy : {best_result.fun:.6f}  (expected -1.0)")
print(f"Optimal parameters θ          : {np.round(best_result.x, 4)}")
print(f"Optimization success          : {best_result.success}")
print(f"Optimizer message             : {best_result.message}")